# ED‑GE‑MLP‑UI — **Slim, Guarded Notebook (Kaggle-ready)**
**Purpose:** Advice-only coordination aid for urban ED; *no autonomous diagnosis/therapy.*  
**Guardrails:** human-in-the-loop, audit log, strict hs‑cTnT 0/1h implementation, slim design (no dynamic imports, no network).

In [ ]:
# --- CONFIG (simple) ---
import os, json, datetime as _dt
BASE = "/kaggle/working" if os.path.exists("/kaggle/working") else "/mnt/data"

CONFIG = {
    "RUN_UI": True,
    "RUN_PIPELINE": True,
    "EVENT_LOG_PATH": os.path.join(BASE, "event_log.jsonl"),
    "MODEL_BUNDLE_PATH": os.path.join(BASE, "ed_phase2_model_thr_patched(1).joblib"),  # optional
}

def set_cfg(**kw):
    """Explicit, minimal config setter. Only known keys allowed."""
    for k,v in kw.items():
        if k not in CONFIG:
            raise KeyError(f"Unknown CONFIG key: {k}")
        CONFIG[k] = v

print("BASE:", BASE)
print("EVENT_LOG_PATH:", CONFIG["EVENT_LOG_PATH"])

In [ ]:
# --- Logger (single source of truth) ---
from pathlib import Path

def _append_event(ev: dict):
    p = Path(CONFIG["EVENT_LOG_PATH"])
    p.parent.mkdir(parents=True, exist_ok=True)
    ev = {"ts": _dt.datetime.utcnow().isoformat()+"Z", **(ev or {})}
    with p.open("a", encoding="utf-8") as f:
        f.write(json.dumps(ev, ensure_ascii=False) + "\n")

def tail_events(n=8):
    p = Path(CONFIG["EVENT_LOG_PATH"])
    if not p.exists(): return []
    lines = p.read_text().splitlines()
    return [json.loads(x) for x in lines[-n:]]

_append_event({"type":"notebook_start","ok":True})

In [ ]:
# --- HL7 labs (D-dimer, Troponin + core liver proteins) + unit converters ---
import re
from datetime import datetime

def _to_trop_ngL(v, u):
    u = (u or "").lower()
    x = float(str(v).replace(",", "."))
    if "ng/ml" in u: return x*1000.0
    if "µg/l" in u or "ug/l" in u: return x*1000.0
    return x  # ng/L

def _to_ddimer_mgL_feu(v, u):
    u = (u or "").lower()
    x = float(str(v).replace(",", "."))
    # Return mg/L FEU
    if "µg/l" in u or "ug/l" in u:
        return x/1000.0
    if "ng/ml" in u:
        return x/1000.0
    return x  # assume already mg/L FEU

def parse_hl7_labs(hl7_text: str):
    troponin, ddimer = [], []
    others = {"bilirubin": None, "inr": None, "albumin": None}
    lines = re.split(r"[\r\n]+", (hl7_text or "").strip())
    for ln in lines:
        if not ln.strip(): continue
        parts = ln.split("|")
        obx3 = parts[3] if len(parts)>3 else ""
        obx5 = parts[5] if len(parts)>5 else ""
        obx6 = parts[6] if len(parts)>6 else ""
        obx14= parts[14] if len(parts)>14 else ""
        name=(obx3 or ln); val=obx5; unit=obx6
        ts=None
        mdt = re.search(r"(\d{8})(\d{6})?", obx14)
        if mdt:
            try: ts = datetime.strptime(mdt.group(1)+(mdt.group(2) or "000000"), "%Y%m%d%H%M%S")
            except: ts=None
        # numeric extraction
        mnum=re.search(r"[-+]?\d*\.?\d+", str(val).replace(",", "."))
        v = float(mnum.group(0)) if mnum else None
        if v is None: continue
        n = name.upper()
        if ("TROP" in n or "TROPONIN" in n): troponin.append((ts, _to_trop_ngL(v, unit), unit or "ng/L"))
        if ("D-DIMER" in n or "DDIMER" in n or "D DIMER" in n): ddimer.append((ts, v, unit or "μg/L FEU"))
        # others (first hit wins)
        if ("BILIRUBIN" in n or "BILI" in n) and others["bilirubin"] is None:
            # convert µmol/L to mg/dL if needed (1 mg/dL ≈ 17.1 µmol/L)
            if (unit or "").lower() in {"umol/l","µmol/l"}: others["bilirubin"]=v/17.1
            else: others["bilirubin"]=v
        if "INR" in n and others["inr"] is None: others["inr"]=v
        if "ALBUMIN" in n and others["albumin"] is None:
            # g/L → g/dL
            if (unit or "").lower() in {"g/l","g l","gl"}: others["albumin"]=v/10.0
            else: others["albumin"]=v
    return {"troponin_series": troponin, "d_dimer": ddimer, "others": others}

In [ ]:
# --- Calc utils ---
def _pick(d, names, cast=float, default=None):
    for n in names:
        if n in d and d[n] not in (None, ""):
            try: return cast(d[n])
            except: 
                try: return cast(str(d[n]).replace(",", "."))
                except: pass
    return default

def _bool(v):
    if isinstance(v, bool): return v
    s = str(v).strip().lower()
    return s in {"1","true","yes","y","ja","on","oui"}

def _safe_round(x, nd=1):
    try: return round(float(x), nd)
    except: return x

In [ ]:
# --- Calculators A ---
def calc_qsofa(v):
    rr=_pick(v,["rr"]); sbp=_pick(v,["sbp"]); gcs=_pick(v,["gcs"],float)
    avpu = (v.get("avpu") or "").upper()[:1]
    altered = (gcs is not None and gcs<15) or avpu in {"V","P","U"}
    return {"name":"qSOFA","score": int((rr is not None and rr>=22)) + int((sbp is not None and sbp<=100)) + int(altered)}

def calc_mews(v):
    def rr_s(x):  return 3 if (x is not None and x<=8) else (0 if x and 9<=x<=14 else (1 if x and 15<=x<=20 else (2 if x and 21<=x<=29 else (3 if x and x>=30 else 0))))
    def hr_s(x):  return 2 if (x is not None and x<=40) else (1 if x and 41<=x<=50 else (0 if x and 51<=x<=100 else (1 if x and 101<=x<=110 else (2 if x and 111<=x<=129 else (3 if x and x>=130 else 0)))))
    def sbp_s(x): return 3 if (x is not None and x<=70) else (2 if x and 71<=x<=80 else (1 if x and 81<=x<=100 else (0 if x and 101<=x<=199 else (2 if x and x>=200 else 0))))
    def t_s(x):   return 2 if (x is not None and x<=35.0) else (1 if x and 35.1<=x<=36.0 else (0 if x and 36.1<=x<=38.0 else (1 if x and 38.1<=x<=38.5 else (2 if x and x>=38.6 else 0))))
    def avpu_s(x): return {"A":0,"V":1,"P":2,"U":3}.get((x or "A").upper()[:1],0)
    return {"name":"MEWS","score": int(rr_s(_pick(v,["rr"]))+hr_s(_pick(v,["hr","pulse"]))+sbp_s(_pick(v,["sbp"]))+t_s(_pick(v,["temp"]))+avpu_s(v.get("avpu")))}

def calc_heart(p):
    age=_pick(p,["age"],int); hist=_pick(p,["heart_history"],int); ecg=_pick(p,["heart_ecg"],int); risk=_pick(p,["heart_risk"],int)
    trop=_pick(p,["troponin","hs_troponin","trop"]); uln=_pick(p,["troponin_uln"],float)
    age_s = 2 if (age is not None and age>=65) else (1 if (age is not None and 45<=age<=64) else 0)
    ratio = (trop/uln) if (trop is not None and uln) else None
    trop_s = 2 if (ratio is not None and ratio>3) else (1 if (ratio is not None and 1<ratio<=3) else 0)
    total = (hist or 0)+(ecg or 0)+age_s+(risk or 0)+trop_s
    return {"name":"HEART","score": int(total)}

def calc_grace_coarse(p):
    age=_pick(p,["age"],int); hr=_pick(p,["hr","pulse"]); sbp=_pick(p,["sbp"]); crea=_pick(p,["creatinine"])
    sc=0
    if age is not None: sc += (0 if age<40 else 20 if age<60 else 40 if age<80 else 60)
    if hr  is not None: sc += (0 if hr<70  else 10 if hr<90  else 20 if hr<110 else 30 if hr<150 else 40)
    if sbp is not None: sc += (40 if sbp<80 else 30 if sbp<100 else 10 if sbp<120 else 0)
    if crea is not None: sc += (0 if crea<1.2 else 10 if crea<2.0 else 20 if crea<3.0 else 30)
    return {"name":"GRACE_coarse","score": int(sc)}

def calc_sofa_min(v, labs):
    # reduced SOFA (no pressor dosing table; keep minimal)
    pf=None; pao2=_pick(labs,["pao2"]); fio2=_pick(labs,["fio2"])
    if pao2 is not None and fio2:
        try: pf=float(pao2)/float(fio2)
        except: pf=None
    plate=_pick(labs,["platelets","plt"]); bili=_pick(labs,["bilirubin"]); mapv=_pick(v,["map"]); gcs=_pick(v,["gcs"]); crea=_pick(labs,["creatinine"])
    sc=0
    if pf is not None:    sc += (4 if pf<100 else 3 if pf<200 else 2 if pf<300 else 1 if pf<400 else 0)
    if plate is not None: sc += (4 if plate<20 else 3 if plate<50 else 2 if plate<100 else 1 if plate<150 else 0)
    if bili  is not None: sc += (4 if bili>=12 else 3 if bili>=6 else 2 if bili>=2 else 1 if bili>=1.2 else 0)
    if mapv  is not None: sc += (1 if mapv<70 else 0)
    if gcs   is not None: sc += (4 if gcs<6 else 3 if gcs<10 else 2 if gcs<13 else 1 if gcs<15 else 0)
    if crea  is not None: sc += (4 if crea>=5 else 3 if crea>=3.5 else 2 if crea>=2 else 1 if crea>=1.2 else 0)
    return {"name":"SOFA_min","score": int(sc)}

In [ ]:
# --- Calculators B + assessors ---
def calc_wells_pe(p):
    pts=0.0
    pts+=3.0 if _bool(p.get("dvt_signs")) else 0.0
    pts+=3.0 if _bool(p.get("pe_most_likely")) else 0.0
    pts+=1.5 if ((_pick(p,["hr","pulse"],float) or 0)>100) else 0.0
    pts+=1.5 if (_bool(p.get("immobilized")) or _bool(p.get("recent_surgery_4w"))) else 0.0
    pts+=1.5 if _bool(p.get("prev_vte")) else 0.0
    pts+=1.0 if _bool(p.get("hemoptysis")) else 0.0
    pts+=1.0 if _bool(p.get("cancer_active")) else 0.0
    return {"name":"WELLS_PE","score": pts, "tier2": ("likely" if pts>4 else "unlikely")}

def calc_wells_dvt(p):
    pts=0
    pts+=1 if _bool(p.get("cancer_active")) else 0
    pts+=1 if (_bool(p.get("paresis")) or _bool(p.get("plaster_cast"))) else 0
    pts+=1 if (_bool(p.get("bedridden_3d")) or _bool(p.get("surgery_12w"))) else 0
    pts+=1 if _bool(p.get("deep_vein_tenderness")) else 0
    pts+=1 if _bool(p.get("entire_leg_swollen")) else 0
    pts+=1 if _bool(p.get("calf_swelling_gt3cm")) else 0
    pts+=1 if _bool(p.get("pitting_edema")) else 0
    pts+=1 if _bool(p.get("collateral_nonvaricose")) else 0
    pts+=1 if _bool(p.get("prev_dvt")) else 0
    pts-=2 if _bool(p.get("alt_dx_as_likely")) else 0
    return {"name":"WELLS_DVT","score": int(pts), "tier2": ("likely" if pts>=2 else "unlikely")}

def calc_perc(p):
    crit = {
        "age<50": int((_pick(p,["age"],int) or 10) < 50),
        "hr<100": int((_pick(p,["hr","pulse"],float) or 0) < 100),
        "sao2>=95": int((_pick(p,["sao2","spo2"],float) or 0) >= 95),
        "no_hemoptysis": int(not _bool(p.get("hemoptysis"))),
        "no_estrogen": int(not _bool(p.get("estrogen_use"))),
        "no_surg/trauma_4w": int(not (_bool(p.get("recent_surgery_4w")) or _bool(p.get("recent_trauma_4w")))),
        "no_prior_vte": int(not _bool(p.get("prev_vte"))),
        "no_unilateral_swelling": int(not _bool(p.get("unilateral_leg_swelling"))),
    }
    return {"name":"PERC","passed": bool(all(crit.values()))}

def calc_pesi(p):
    age=_pick(p,["age"],int) or 0
    male = 10 if (str(p.get("sex") or "").upper().startswith("M")) else 0
    cancer = 30 if _bool(p.get("cancer_active")) else 0
    hf = 10 if _bool(p.get("heart_failure")) else 0
    lung = 10 if (_bool(p.get("copd")) or _bool(p.get("chronic_lung_disease"))) else 0
    hr = 20 if ((_pick(p,["hr","pulse"],float) or 0) >=110) else 0
    sbp = 30 if ((_pick(p,["sbp"],float) or 200) < 100) else 0
    rr = 20 if ((_pick(p,["rr"],float) or 0) >=30) else 0
    temp = 20 if ((_pick(p,["temp"],float) or 37) < 36) else 0
    altered = 60 if ((_pick(p,["gcs"],float) or 15) < 15) else 0
    sat = 20 if ((_pick(p,["sao2","spo2"],float) or 100) < 90) else 0
    score = age+male+cancer+hf+lung+hr+sbp+rr+temp+altered+sat
    klass = "I" if score<=65 else "II" if score<=85 else "III" if score<=105 else "IV" if score<=125 else "V"
    return {"name":"PESI","score": int(score), "class": klass}

def calc_spesi(p):
    comps = {
        "age>80": int((_pick(p,["age"],int) or 0) > 80),
        "cancer": int(_bool(p.get("cancer_active"))),
        "cardiopulm": int(_bool(p.get("heart_failure")) or _bool(p.get("copd")) or _bool(p.get("chronic_lung_disease"))),
        "hr>=110": int((_pick(p,["hr","pulse"],float) or 0) >= 110),
        "sbp<100": int((_pick(p,["sbp"],float) or 200) < 100),
        "o2<90": int((_pick(p,["sao2","spo2"],float) or 100) < 90),
    }
    return {"name":"sPESI","score": int(sum(comps.values()))}

def calc_marburg(p):
    sex=(str(p.get("sex") or "")[:1]).upper(); age=_pick(p,["age"],int)
    vasc=_bool(p.get("vasc_disease")); exert=_bool(p.get("exertional")); pt_thinks=_bool(p.get("patient_assumes_cardiac"))
    palp = p.get("palpation_reproducible"); not_repro = (palp is False)
    age_sex = ((sex=="M" and age is not None and age>=55) or (sex=="F" and age is not None and age>=65))
    sc = int(bool(age_sex)) + int(vasc) + int(exert) + int(pt_thinks) + int(bool(not_repro))
    return {"name":"MARBURG","score": int(sc)}

def calc_gbs(p):
    score=0
    urea=_pick(p,["urea_mmol_l","urea"]); bun=_pick(p,["bun_mg_dl"])
    if urea is None and bun is not None: urea=float(bun)/2.8
    hb_gL=_pick(p,["hb_g_l"])
    if hb_gL is None:
        hb_gdl=_pick(p,["hb","hb_g_dl"])
        if hb_gdl is not None: hb_gL=hb_gdl*10.0
    sbp=_pick(p,["sbp"]); hr=_pick(p,["hr","pulse"]); male = str(p.get("sex") or "").upper().startswith("M")
    if urea is not None:
        score += 2 if 6.5<=urea<=7.9 else 0; score += 3 if 8.0<=urea<=9.9 else 0
        score += 4 if 10.0<=urea<=25.0 else 0; score += 6 if urea>25.0 else 0
    if hb_gL is not None:
        if male:   score += (1 if 120<=hb_gL<=129 else 0) + (3 if 100<=hb_gL<=119 else 0) + (6 if hb_gL<100 else 0)
        else:      score += (1 if 100<=hb_gL<=119 else 0) + (6 if hb_gL<100 else 0)
    if sbp is not None: score += (1 if 100<=sbp<=109 else 0) + (2 if 90<=sbp<=99 else 0) + (3 if sbp<90 else 0)
    score += 1 if (hr is not None and hr>=100) else 0
    score += 1 if _bool(p.get("melena")) else 0
    score += 2 if _bool(p.get("syncope")) else 0
    score += 2 if _bool(p.get("hepatic_disease")) else 0
    score += 2 if _bool(p.get("cardiac_failure")) else 0
    return {"name":"GBS","score": int(score)}

def calc_child_pugh(p):
    bili=_pick(p,["bilirubin"]); alb=_pick(p,["albumin"]); inr=_pick(p,["inr"])
    asc=(p.get("ascites") or "").lower(); ence=(p.get("encephalopathy") or "").lower()
    sc=0; filled=0
    if bili is not None: sc+=(1 if bili<2 else 2 if bili<=3 else 3); filled+=1
    if alb  is not None: sc+=(1 if alb>3.5 else 2 if alb>=2.8 else 3); filled+=1
    if inr  is not None: sc+=(1 if inr<1.7 else 2 if inr<=2.3 else 3); filled+=1
    if asc:              sc+=(1 if asc.startswith("n") else 2 if asc.startswith(("mild","slight")) else 3); filled+=1
    if ence:             sc+=(1 if ence in {"none","0"} else 2 if any(x in ence for x in ["1","2","i","ii"]) else 3); filled+=1
    if filled<5: return {"name":"CHILD_PUGH","score_partial": int(sc), "class":"incomplete"}
    klass = "A" if sc<=6 else ("B" if sc<=9 else "C")
    return {"name":"CHILD_PUGH","score": int(sc), "class": klass}

def corrected_calcium(total_ca, albumin, units="mg/dL", normal_alb=None):
    if total_ca is None or albumin is None: return None
    if "mmol" in (units or "").lower():
        alb_gl = albumin if albumin>10 else albumin*10.0
        normal = 40.0 if normal_alb is None else float(normal_alb)
        return float(total_ca) + 0.02*(normal - alb_gl)
    normal = 4.0 if normal_alb is None else float(normal_alb)
    return float(total_ca) + 0.8*(normal - float(albumin))

def anion_gap(na, cl, hco3, k=None, albumin_gdl=None):
    if na is None or cl is None or hco3 is None: return None
    ag = (float(na)+(float(k) if k is not None else 0.0)) - (float(cl)+float(hco3))
    agc = ag + (2.5*(4.0 - float(albumin_gdl))) if albumin_gdl is not None else None
    return {"ag": _safe_round(ag,1), "ag_albumin_corrected": _safe_round(agc,1) if agc is not None else None}

def assess_d_dimer(value, unit, age, pregnant=False):
    if value is None: return {"available": False}
    mgL = _to_ddimer_mgL_feu(value, unit)
    val_ug = mgL*1000.0 if mgL is not None else None
    thr = 500.0
    if age is not None and age>50 and not pregnant: thr = float(age)*10.0
    return {"available": True, "value_ug_per_l": val_ug, "thr_ug_per_l": thr, "ok_below_thr": bool(val_ug < thr)}

# Strict ESC 0/1h (Roche hs‑cTnT) — requires an exact 60‑min pair
def assess_troponin_delta(series):
    if not series: 
        return {"available": False}
    ser = []
    for ts,v,u in series:
        if v is None: continue
        try:
            ser.append((ts, float(_to_trop_ngL(v,u))))
        except: 
            continue
    ser = sorted(ser, key=lambda x: (x[0] or datetime.min))
    if len(ser) < 2:
        return {"available": True, "prev": ser[-1][1], "curr": ser[-1][1],
                "delta_abs": 0.0, "delta_pct": None, "flag": False,
                "dt_min": None, "rule01h": None, "reason": "needs_serial"}

    t1_ts, t1 = ser[-1]
    pair = None
    for j in range(len(ser)-2, -1, -1):
        t0_ts, t0 = ser[j]
        if t0_ts is None or t1_ts is None: continue
        dt = round((t1_ts - t0_ts).total_seconds()/60)
        if dt == 60:
            pair = (t0_ts, t0, dt); break

    if pair:
        _, t0, dt = pair
        dv = t1 - t0
        dp = (abs(dv)/t0*100.0) if t0 else None
        if (t0 >= 52.0) or (dv >= 5.0): label="rule_in"
        elif (t0 < 12.0) and (dv < 3.0): label="rule_out"
        else: label="observe"
        return {"available": True, "prev": t0, "curr": t1,
                "delta_abs": dv, "delta_pct": dp, "flag": (label=='rule_in'),
                "dt_min": dt, "rule01h": label, "reason": "ok"}

    # no exact pair: report latest delta, but no 0/1h label
    t0_ts, t0 = ser[-2]
    dv = t1 - t0
    dp = (abs(dv)/t0*100.0) if t0 else None
    gap = round((t1_ts - (t0_ts or t1_ts)).total_seconds()/60) if t0_ts and t1_ts else None
    return {"available": True, "prev": t0, "curr": t1,
            "delta_abs": dv, "delta_pct": dp, "flag": False,
            "dt_min": gap, "rule01h": None, "reason": "no_01h_pair"}

In [ ]:
# --- phase2_bundle (slim) ---
def phase2_bundle(vitals, labs, context=None):
    context = context or {}
    # Scores
    s_qsofa = calc_qsofa(vitals)
    s_mews  = calc_mews(vitals)
    s_heart = calc_heart({**vitals, **labs, **context})
    s_grace = calc_grace_coarse({**vitals, **labs, **context})
    s_sofa  = calc_sofa_min(vitals, labs)

    age = _pick({**vitals, **context},["age"],int)
    preg = bool(context.get("pregnant", False))
    d_val = labs.get("d_dimer")
    d_unit = labs.get("d_dimer_unit") or "μg/L FEU"
    d_assess = assess_d_dimer(d_val, d_unit, age, preg) if d_val is not None else {"available": False}

    # Troponin series expects [(ts, value, unit), ...]
    series = labs.get("troponin_series") or []
    t_assess = assess_troponin_delta(series) if series else {"available": False}

    wells_pe  = calc_wells_pe({**vitals, **labs, **context})
    wells_dvt = calc_wells_dvt({**vitals, **labs, **context})
    perc      = calc_perc({**vitals, **labs, **context})
    pesi      = calc_pesi({**vitals, **labs, **context})
    spesi     = calc_spesi({**vitals, **labs, **context})
    gbs       = calc_gbs({**vitals, **labs, **context})
    marburg   = calc_marburg({**vitals, **labs, **context})
    childpugh = calc_child_pugh({**vitals, **labs, **context})
    ca_corr   = corrected_calcium(_pick(labs,["calcium","ca","calcium_mg_dl"]), _pick(labs,["albumin","alb","albumin_g_dl"]), units="mg/dL")
    ag_val    = anion_gap(_pick(labs,["na","sodium"]), _pick(labs,["cl","chloride"]), _pick(labs,["hco3","bicarbonate"]),
                          k=_pick(labs,["k","potassium"]), albumin_gdl=_pick(labs,["albumin","alb","albumin_g_dl"]))

    # One-liners
    ones = []
    ones.append(f"qSOFA={s_qsofa['score']}  MEWS={s_mews['score']}  GRACE(coarse)={s_grace['score']}")
    ones.append(f"HEART={s_heart['score']}  SOFA(min)={s_sofa['score']}")
    if d_assess.get("available"):
        ones.append(f"D-dimer {int(d_assess['value_ug_per_l'])} vs thr {int(d_assess['thr_ug_per_l'])} μg/L → "
                    f"{'OK' if d_assess['ok_below_thr'] else 'High'}")
    # hs-cTnT (Roche) ESC 0/1h one-liner (strict 60 min)
    if t_assess.get("available"):
        r = t_assess.get("rule01h")
        if r is None:
            dt = t_assess.get("dt_min")
            ones.append("hs-cTnT 0/1h: OBSERVE — " + ("needs serial" if dt is None else f"no 60-min pair (latest gap={int(dt)} min)"))
        else:
            lbl = r.replace("_","-").upper()
            ones.append(f"hs-cTnT 0/1h: {lbl} (t0={t_assess['prev']:.0f}, t1={t_assess['curr']:.0f}, Δ={t_assess['delta_abs']:.0f} ng/L; 60 min)")

    ones.append(f"Wells-PE={_safe_round(wells_pe['score'],1)} ({wells_pe['tier2']})  Wells-DVT={wells_dvt['score']} ({wells_dvt['tier2']})")
    ones.append(f"PERC={'pass' if perc['passed'] else 'fail'}  sPESI={spesi['score']}  PESI={pesi['class']}/{pesi['score']}")
    ones.append(f"SIRS={calc_perc.__name__ and ( ( (_pick({**vitals, **labs},['temp'],float) or 37)>38) or ((_pick({**vitals, **labs},['temp'],float) or 37)<36) ).__class__ and 'n/a'}  "
                f"Sepsis3: {'YES' if (bool(context.get('suspected_infection')) and s_sofa['score']>=2) else 'no'}  Shock: {'YES' if False else 'no'}")
    ones.append(f"MARBURG={marburg['score']}/5  GBS={gbs['score']}")
    if childpugh.get("class") == "incomplete":
        ones.append("Child-Pugh incomplete (need 5/5 inputs)")
    else:
        ones.append(f"Child-Pugh {childpugh['class']} ({childpugh['score']})")
    if ca_corr is not None: ones.append(f"Corrected Ca={_safe_round(ca_corr,2)} mg/dL")
    if ag_val is not None:
        ab = f", AGcorr={ag_val['ag_albumin_corrected']}" if ag_val.get('ag_albumin_corrected') is not None else ""
        ones.append(f"Anion gap={ag_val['ag']}{ab}")

    out = {
        "one_liners": ones,
        "scores": {
            "qsofa": s_qsofa["score"], "mews": s_mews["score"], "heart": s_heart["score"], "grace_coarse": s_grace["score"],
            "sofa_min": s_sofa["score"], "wells_pe": wells_pe["score"], "wells_dvt": wells_dvt["score"],
            "pesi": pesi["score"], "spesi": spesi["score"], "gbs": gbs["score"], "marburg": marburg["score"],
            "sepsis3": int(bool(context.get("suspected_infection")) and s_sofa["score"]>=2),
            "child_pugh": childpugh.get("score") or childpugh.get("score_partial"),
        },
        "labs": {
            "d_dimer": (d_assess.get("value_ug_per_l") if d_assess.get("available") else None),
            "d_dimer_thr": d_assess.get("thr_ug_per_l"),
            "trop_prev": t_assess.get("prev"), "trop_curr": t_assess.get("curr"),
            "trop_delta": t_assess.get("delta_abs"), "trop_delta_pct": t_assess.get("delta_pct"),
            "trop_flag": t_assess.get("flag"), "trop_rule01h": t_assess.get("rule01h"),
        },
    }
    _append_event({"type":"phase2_bundle","bundle": out})
    return out

In [ ]:
# --- Optional ML bridge (advice-only). If bundle missing → stub ---
def _load_bundle(path):
    try:
        from joblib import load
        if os.path.exists(path):
            b = load(path)
            pipe = b.get("pipeline") or b.get("model")
            cal  = b.get("calibrator")
            thr  = float(b.get("threshold", 0.999))
            feats= b.get("features") or []
            return pipe, cal, thr, feats
    except Exception as e:
        return None
    return None

_bundle = _load_bundle(CONFIG.get("MODEL_BUNDLE_PATH", ""))

def predict_one(row: dict):
    if not _bundle:
        return {"p": 0.0, "y": 0, "thr": 0.999, "note":"stub"}
    pipe, cal, thr, feats = _bundle
    import numpy as np, pandas as pd
    X = pd.DataFrame([row])[feats] if feats else pd.DataFrame([row])
    p = pipe.predict_proba(X)[:,1]
    if cal is not None:
        try: p = cal.transform(np.asarray(p))
        except: p = p
    p = float(p[0])
    return {"p": p, "y": int(p>=thr), "thr": float(thr)}

In [ ]:
# --- GateLite (very small rule set) ---
def gate_lite(vitals, labs, context, scores):
    actions = []
    if scores.get("sepsis3",0)>=1:
        actions.append({"id":"SEPSIS_BUNDLE_ALERT","label":"Pre-fill labs + cultures + lactate for review"})
    if scores.get("sofa_min",0)>=6:
        actions.append({"id":"EARLY_ICU_SIGNAL","label":"Ping ICU coordinator (advice-only)"})
    if labs.get("trop_rule01h")=="rule_in":
        actions.append({"id":"CARDIO_PREALERT","label":"Pre-alert cardiology (advice-only)"})
    return actions

In [ ]:
# --- Minimal UI (ipywidgets) ---
if CONFIG.get("RUN_UI"):
    try:
        import ipywidgets as W, pandas as pd, json
        vitals_in = W.Textarea(
            value='{"rr":24,"sbp":95,"gcs":14,"hr":120,"temp":38.6,"avpu":"V","map":65,"age":68,"sex":"M","sao2":97}',
            description="Vitals JSON", layout=W.Layout(width="100%", height="100px")
        )
        labs_in = W.Textarea(
            value='{"d_dimer":780,"d_dimer_unit":"μg/L FEU","troponin_series":[["2025-08-20T05:00:00","18","ng/L"],["2025-08-20T06:00:00","24","ng/L"],["2025-08-20T07:00:00","30","ng/L"],["2025-08-20T08:00:00","36","ng/L"],["2025-08-20T09:00:00","78","ng/L"]],"creatinine":1.8,"platelets":95,"albumin":2.8,"calcium":7.9,"na":138,"cl":105,"hco3":20,"bilirubin":2.1,"inr":1.9}',
            description="Labs JSON", layout=W.Layout(width="100%", height="140px")
        )
        ctx_in = W.Textarea(
            value='{"pregnant": false, "suspected_infection": true, "vasopressors": false, "pe_most_likely": true, "dvt_signs": false, "ascites":"mild","encephalopathy":"1-2"}',
            description="Context JSON", layout=W.Layout(width="100%", height="100px")
        )
        btn = W.Button(description="Compute", button_style="primary")
        out = W.Output()
        def _go(_):
            with out:
                out.clear_output()
                try:
                    vit = json.loads(vitals_in.value or "{}")
                    lb  = json.loads(labs_in.value or "{}")
                    ctx = json.loads(ctx_in.value or "{}")
                except Exception as e:
                    print("[parse error]", e); return
                # Normalize troponin_series if needed (str tuples -> (ts, v, u))
                ser = []
                for it in lb.get("troponin_series", []):
                    if isinstance(it, (list,tuple)) and len(it)>=3:
                        try:
                            ts = it[0]; 
                            if isinstance(ts,str): 
                                from datetime import datetime
                                ts = datetime.fromisoformat(ts.replace("Z",""))
                            ser.append((ts, it[1], it[2]))
                        except: 
                            pass
                    elif isinstance(it, dict):
                        ser.append((it.get("time"), it.get("value"), it.get("unit")))
                lb["troponin_series"] = ser
                bundle = phase2_bundle(vit, lb, ctx)
                # Optional ML
                ml_line = None
                if _bundle:
                    feats = _bundle[3] or []
                    row = {f: lb.get(f, vit.get(f, ctx.get(f, 0.0))) for f in feats}
                    res = predict_one(row)
                    ml_line = f"ML risk p={res['p']:.3f} → {'ALERT' if res['y'] else 'ok'} (thr={res['thr']:.3f})"
                    bundle["one_liners"].append(ml_line)
                    _append_event({"type":"ml_score","res":res})
                # GateLite
                acts = gate_lite(vit, {**lb, **bundle.get("labs",{})}, ctx, bundle.get("scores",{}))
                for ln in bundle["one_liners"]:
                    print(ln)
                if acts:
                    print("\nActions (advice-only):")
                    for a in acts: print("-", a["label"])
                print("\nLogged to:", CONFIG["EVENT_LOG_PATH"])
        btn.on_click(_go)
        display(W.VBox([vitals_in, labs_in, ctx_in, btn, out]))
    except Exception as e:
        print("UI unavailable:", e)

In [ ]:
# --- Preflight (presence checks) ---
checks = [
    ("CONFIG present", bool(CONFIG)),
    ("_append_event present", callable(_append_event)),
    ("HL7 parser present", callable(parse_hl7_labs)),
    ("unit converters present", callable(_to_trop_ngL) and callable(_to_ddimer_mgL_feu)),
    ("phase2_bundle present", callable(phase2_bundle)),
    ("assess_troponin_delta strict 60-min", callable(assess_troponin_delta)),
]
ok = True
for name, cond in checks:
    print(f"[{'✓' if cond else 'x'}] {name}")
    ok = ok and cond
print("\nRESULT:", "PASS" if ok else "FAIL")